# Exploración, Limpieza, Inflación de Ceros

1. **Remover los rezagos precalculados (`*_lag_*`)** y verificar la integridad del panel espacio-temporal.
2. **Auditar las variables** para detectar *data leakage* (`rr`, `clasi_rr`, `clasi_rr_no_0`) y entender la granularidad de cada covariable.
3. **Diagnosticar la inflación de ceros y sobredispersión** comparando el conteo de casos contra modelos Poisson y Binomial Negativa.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

In [ ]:
ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

os.chdir(ROOT_DIR)
print(f"Directorio raíz del proyecto: {ROOT_DIR}")

## 1. Carga de datos y eliminación de rezagos precalculados (`*_lag_*`)

El archivo crudo trae rezagos precalculados (`_lag_1` a `_lag_8`) pensados para modelos tabulares.  
Para modelos secuenciales y espacio-temporales (**ST-GNN** y **Filtro de Kalman**), trabajar con la tabla base sin rezagos es mucho mejor porque:

- Nos permite definir ventanas deslizantes $T_{\text{in}}$ y horizontes $H$ dinámicamente sin riesgo de fuga entre splits.
- Si algún modelo de Nivel I (SARIMAX/XGBoost) necesita rezagos, los generaremos de forma explícita y controlada después.

In [ ]:
df_raw = pd.read_csv(ROOT_DIR / "data/raw/full_data.csv")
print("Shape original (con lags):", df_raw.shape)

lag_cols = [c for c in df_raw.columns if "_lag_" in c]
base_cols = [c for c in df_raw.columns if "_lag_" not in c]

print(f"Columnas *_lag_*: {len(lag_cols)} | Columnas base: {len(base_cols)}")
df = df_raw[base_cols].copy()
df.head()

In [ ]:
print("Columnas base (sin lags):")
print(list(df.columns))

## 2. Estructura del panel espacio-temporal (`cantón × semana`) y valores faltantes

Descomponemos `week_canton` (`YYYY-W-CCC`) en `year`, `week`, `canton` y `province` (primer dígito del código cantonal), y verificamos:
- Si el panel está perfectamente balanceado (mismas semanas en todos los cantones).
- Si existen duplicados, semanas 53 o valores nulos.

In [ ]:
df[["year", "week", "canton"]] = df["week_canton"].str.split("-", expand=True).astype(int)
df["province"] = df["canton"] // 100

# Índice temporal continuo (0, 1, ..., T-1) ordenado cronológicamente
yw_grid = (
    df[["year", "week"]]
    .drop_duplicates()
    .sort_values(["year", "week"])
    .reset_index(drop=True)
)
yw_grid["time_idx"] = np.arange(len(yw_grid))
df = df.merge(yw_grid, on=["year", "week"], how="left")
df = df.sort_values(["canton", "time_idx"]).reset_index(drop=True)

n_cantons = df["canton"].nunique()
n_weeks = df["time_idx"].nunique()
semanas_por_canton = df.groupby("canton")["week_canton"].nunique()

print(f"Cantones únicos: {n_cantons}")
print(f"Semanas únicas: {n_weeks} (desde {yw_grid.iloc[0]['year']}-W{yw_grid.iloc[0]['week']:02d} hasta {yw_grid.iloc[-1]['year']}-W{yw_grid.iloc[-1]['week']:02d})")
print(f"Semanas por cantón (min, max): {semanas_por_canton.min()}, {semanas_por_canton.max()}")
print(f"¿Panel balanceado ({n_cantons} x {n_weeks} == {len(df)})?: {n_cantons * n_weeks == len(df)}")
print(f"Duplicados en week_canton: {df['week_canton'].duplicated().sum()}")
print(f"Valores nulos totales: {df.isna().sum().sum()}")
print(f"Filas con semana 53: {(df['week'] == 53).sum()}")

Verificación de valores faltantes, en este caso se prensentarían de manera lineas blancas.

In [ ]:
sns.heatmap(df.isna(), cbar=False)
plt.show()

## 3. Auditoría de *Data Leakage* y granularidad real de las covariables

Antes de modelar, debemos saber qué representa cada columna en el tiempo $t$:
- **`rr`, `clasi_rr`, `clasi_rr_no_0`**: derivan directamente de `casos` en la misma semana $t$ (Riesgo Relativo $RR_{it} = \text{casos}_{it} / E_{it}$). Usarlos como predictores en $t$ sería **fuga directa del target**. Esto ocurre porque casi 15mil observaciones están indicando que son casos ceros, es decir, el modelo no estaría aprendiendo a resolver el modelo, sino que estaría sacando la información de esa variable y el modelo inclinandose por esta.

- Sin embargo, `casos / rr` nos revela $E_{it}$ (los casos esperados bajo tasa nacional uniforme), cuya razón respecto al total semanal es prácticamente constante por cantón (~proporción poblacional).

- También verificaremos qué variables son **estáticas** (`urb`), **globales** (`nino34ssta`), **provinciales** (Google Trends) o **cantonales espacio-temporales** (clima y `ndvi`).

In [ ]:
# Verificar que clasi_rr == 'Bajo' equivale a casos == 0
print("Distribución de rr según clasi_rr:")
display(df.groupby("clasi_rr")["rr"].agg(["count", "min", "median", "max"]))

# Verificar casos esperados implícitos E_it = casos / rr respecto al total nacional semanal
weekly_tot = df.groupby("time_idx")["casos"].transform("sum")
pos = df[df["casos"] > 0].copy()
pos["E_it"] = pos["casos"] / pos["rr"]
pos["pop_share_proxy"] = pos["E_it"] / weekly_tot.loc[pos.index]

print("\nEstabilidad de pop_share_proxy = E_it / total_nacional_t dentro de cada cantón (primeros 8):")
display(pos.groupby("canton")["pop_share_proxy"].agg(["mean", "std", "min", "max"]).head(8))

### Interpretación: ¿Por qué auditamos la granularidad y la varianza por dimensión?

Evaluar en qué ejes (tiempo, provincia o cantón) una variable presenta desviación estándar igual o mayor a cero permite determinar **hasta qué nivel de resolución aporta información real** sobre la dinámica del dengue:

1. **Variables con señal espacio-temporal local (ej. nieve en Noruega vs. Costa Rica):**  
   Si construyéramos un grafo mundial, la variable *nieve* tendría varianza positiva en países nórdicos como Noruega y varianza cero en Costa Rica (es decir dentro del país no aporta). A nivel global aparecería como una variable espacio-temporal, pero **dentro del subgrafo de Costa Rica sería constante (sin poder predictivo local)**. Esto demuestra que es necesario verificar no solo si una variable cambia en algún punto del dataset, sino en qué nodos y niveles espaciales aporta señal efectiva.

2. **Variables constantes en el espacio (ej. tasa constante de criaderos en Centroamérica):**  
   Si en una misma semana todas las unidades geográficas comparten exactamente el mismo valor (varianza espacial igual a cero), la variable **no ayuda a explicar por qué un cantón presenta mayor incidencia que otro**. No obstante, si su valor cambia a lo largo de las semanas, sí puede capturar **dinámicas temporales compartidas** (como ocurre en nuestros datos con el índice global `nino34ssta` a escala nacional, o con las búsquedas de Google Trends a escala provincial).

**Conclusión para el modelado:**  
Una desviación estándar igual a cero en una dimensión indica que la variable es redundante en ese eje. Esta parte nos permite asignar cada grupo de covariables al rol correcto dentro de la arquitectura **ST-GNN + Filtro de Kalman**:
* **Estáticas por cantón** (`urb`): atributos fijos de cada nodo del grafo.
* **Globales semanales** (`nino34ssta`): contexto macroclimático compartido por todos los nodos.
* **Provinciales semanales** (Google Trends): señales de comportamiento poblacional a escala regional.
* **Espacio-temporales** (clima y `ndvi`): covariables dinámicas locales por cantón y semana, donde sí tiene sentido evaluar rezagos biológicos.

In [ ]:
# Verificar variación espacial (entre cantones cada semana) vs temporal (entre semanas cada cantón)
exclude = {"year", "week", "canton", "province", "time_idx"}
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]

gran_rows = []
for col in num_cols:
    std_between_cantons = df.groupby("time_idx")[col].std().fillna(0).max()
    std_within_province = df.groupby(["time_idx", "province"])[col].std().fillna(0).max()
    std_over_time = df.groupby("canton")[col].std().fillna(0).max()
    if std_over_time == 0:
        nivel = "Estática por cantón"
    elif std_between_cantons == 0:
        nivel = "Global semanal (nacional)"
    elif std_within_province == 0:
        nivel = "Provincial semanal"
    else:
        nivel = "Espacio-temporal (cantón x semana)"
    gran_rows.append({"variable": col, "nivel": nivel, "min": df[col].min(), "max": df[col].max()})

gran_df = pd.DataFrame(gran_rows)
display(gran_df)

## 4. Análisis profundo de Inflación de Ceros y Sobredispersión

Casi el **60%** de las observaciones semanales tienen `casos == 0`. Por eso `MSE` o una regresión gaussiana directa no son adecuadas.

Pero en estadística de conteos hay una pregunta fundamental antes de elegir la función de pérdida (Poisson vs Binomial Negativa vs Zero-Inflated Negative Binomial / Hurdle):
> **¿Los ceros observados son "ceros en exceso" estructurales, o son consecuencia de la sobredispersión extrema y la heterogeneidad entre cantones?**

Comparamos la proporción observada $P(Y=0)$ contra lo que predicen:
- **Poisson($\mu$)**: $P(Y=0) = e^{-\mu}$
- **Binomial Negativa($\mu, r$)** por método de momentos ($r = \frac{\mu^2}{\sigma^2 - \mu}$): $P(Y=0) = \left(\frac{r}{r + \mu}\right)^r$

En este caso estamos analizando cómo se comportan a nivel país el número promedio de casos de dengue en Costa Rica, con el fin de analizar mi función de perdida. En este punto observamos que los datos a nivel país tienen tenemos una cantidad del 59% casos que se van a reportar 0, con los datos empiricos:

1) La Poisson estima observar que la cantidad de casos de dengue que son cero es de 2.83%.
2) La Binomial Negativa se puede observar se observa que estima que el número de casos de cero debería estar en 74.01%, en este caso sobre estimó el porcentaje.

En este punto sabemos que hay una sobredispersión severa, pero puede ser que esa cantidad de ceros que estamos observando sea a nivel estructural, ya que hay cantones/provincias que son muy propensas a reportar dengue.

Aquí hay que tomar un punto muy importante, los casos de dengue pueden ser estructurales, entonces puede ser un error verlos a nivel local, por tanto, hay que cambiar esta percepción a nivel cantón y ver que si dentro de cantón Poisson sigue sufriendo o lo mejor es usar Binomial Negativa.

In [ ]:
y = df["casos"].to_numpy(dtype=float)
mean_g = y.mean()
var_g = y.var(ddof=1)
disp_g = var_g / mean_g
r_g = (mean_g ** 2) / (var_g - mean_g)

p0_obs_g = (y == 0).mean()
p0_pois_g = np.exp(-mean_g)
p0_nb_g = (r_g / (r_g + mean_g)) ** r_g

print(f"Observaciones totales: {len(y):,}")
print(f"Media global: {mean_g:.4f} | Varianza global: {var_g:.2f} | Índice de dispersión (Var/Media): {disp_g:.2f}")
print(f"P(Y=0) observada       : {p0_obs_g*100:.2f}%")
print(f"P(Y=0) esperada Poisson: {p0_pois_g*100:.2f}%")
print(f"P(Y=0) esperada NegBin : {p0_nb_g*100:.2f}% (r={r_g:.4f})")

En este punto observamos que a nivel de Poisson tenemos que hay cantones que tiene almenos un 5% de exceso de casos vistos por Poisson, lo que implica que el modelo a nivel de cantón sigue teniendo problemas para comprender el patrón de los casos registrados. En cambio, Binomial Negativa solo cuenta con 2 casos que apenas tuvieron un exceso de ceros cuando se realizar la estimación de cuántos ceros debería de tener tomando en cuenta este modelo.

In [ ]:
# Diagnóstico cantón por cantón
diag_rows = []
for cid, grp in df.groupby("canton"):
    yc = grp["casos"].to_numpy(dtype=float)
    m = yc.mean()
    v = yc.var(ddof=1)
    p0_obs = (yc == 0).mean()
    p0_pois = np.exp(-m)
    if v > m and m > 0:
        rc = (m ** 2) / (v - m)
        p0_nb = (rc / (rc + m)) ** rc
    else:
        rc = np.inf
        p0_nb = p0_pois
    diag_rows.append({
        "canton": cid,
        "province": grp["province"].iloc[0],
        "urb": grp["urb"].iloc[0],
        "total_casos": int(yc.sum()),
        "mean": m,
        "var": v,
        "dispersion_idx": v / m if m > 0 else np.nan,
        "p0_obs": p0_obs,
        "p0_poisson": p0_pois,
        "p0_negbin": p0_nb,
        "excess_vs_poisson": p0_obs - p0_pois,
        "excess_vs_negbin": p0_obs - p0_nb,
    })

canton_diag = pd.DataFrame(diag_rows)
print("Cantones con >5% exceso de ceros frente a Poisson:", (canton_diag["excess_vs_poisson"] > 0.05).sum(), "/ 82")
print("Cantones con >5% exceso de ceros frente a Binomial Negativa:", (canton_diag["excess_vs_negbin"] > 0.05).sum(), "/ 82")
print("Mediana de (p0_obs - p0_negbin):", round(canton_diag["excess_vs_negbin"].median(), 4))

En la siguiente tabla observamos que Poisson indica que por cantón debería una proporción de ceros de 0%, en cambio la binomial negativa indica que es 16.8%, y en realidad hay una cantidad 16.8% (aproximandamente), como observamos para el primer cantón el modelo mejoró su aproximación.

In [ ]:
print("Top 10 cantones por casos totales:")
display(canton_diag.sort_values("total_casos", ascending=False).head(10).round(3))

In [ ]:
print("Top 10 cantones con mayor proporción de ceros:")
display(canton_diag.sort_values("p0_obs", ascending=False).head(10).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(canton_diag["p0_poisson"], canton_diag["p0_obs"], alpha=0.75, color="#d62728", label="vs Poisson")
ax.scatter(canton_diag["p0_negbin"], canton_diag["p0_obs"], alpha=0.80, color="#2ca02c", marker="^", label="vs Binomial Negativa")
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Ajuste exacto (y = x)")
ax.set_title("Proporción de Ceros por Cantón: Observada vs Esperada")
ax.set_xlabel("P(Y = 0) esperada por el modelo")
ax.set_ylabel("P(Y = 0) observada")
ax.legend()

ax = axes[1]
pos_c = canton_diag[canton_diag["mean"] > 0]
sc = ax.scatter(pos_c["mean"], pos_c["var"], c=pos_c["p0_obs"] * 100, cmap="viridis", s=50, alpha=0.85)
m_grid = np.logspace(np.log10(pos_c["mean"].min()), np.log10(pos_c["mean"].max()), 100)
ax.plot(m_grid, m_grid, "k--", lw=1.2, label="Poisson (Var = Media)")
ax.plot(m_grid, m_grid + 2.0 * (m_grid ** 2), color="#ff7f0e", ls="-.", lw=1.4, label="Binomial Negativa (r=0.5)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Sobredispersión Cantonal: Media vs Varianza (log-log)")
ax.set_xlabel("Media semanal de casos")
ax.set_ylabel("Varianza semanal de casos")
ax.legend()
plt.colorbar(sc, ax=ax, label="% de ceros")

fig.tight_layout()
fig_zi_path = ROOT_DIR / "data/processed/eda/02_zero_inflation_y_sobredispersion.png"
fig_zi_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_zi_path, dpi=150)
plt.show()

## 5. Dinámica temporal, estacionalidad y diseño del split (`Train / Val / Test`)

En series espacio-temporales **nunca** se divide aleatoriamente; siempre se divide en orden cronológico para evitar fuga del futuro.
Analizamos la curva epidémica anual (especialmente la onda 2023–2024) y la estacionalidad por semana epidemiológica 1–52.

In [ ]:
yearly = df.groupby("year")["casos"].agg(
    observaciones="count",
    pct_zeros=lambda s: round((s == 0).mean() * 100, 2),
    media=lambda s: round(s.mean(), 2),
    total="sum",
    max="max",
)
display(yearly)

weekly_nat = df.groupby(["time_idx", "year", "week"], as_index=False)["casos"].sum()
val_start = int(weekly_nat.loc[weekly_nat["year"] == 2024, "time_idx"].min())
test_start = int(weekly_nat.loc[weekly_nat["year"] == 2025, "time_idx"].min())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
ax.plot(weekly_nat["time_idx"], weekly_nat["casos"], color="#1f77b4", lw=1.8)
ax.axvline(val_start, color="#ff7f0e", ls="--", lw=1.6, label="Inicio Val (2024)")
ax.axvline(test_start, color="#d62728", ls="--", lw=1.6, label="Inicio Test (2025)")
year_ticks = weekly_nat.groupby("year")["time_idx"].first()
ax.set_xticks(year_ticks.values)
ax.set_xticklabels([str(y) for y in year_ticks.index])
ax.set_title("Curva Epidémica Nacional y Split Temporal Propuesto")
ax.set_xlabel("Año")
ax.set_ylabel("Casos semanales (82 cantones)")
ax.grid(False)
ax.legend()

seas = df.groupby("week", as_index=False).agg(
    mean_casos=("casos", "mean"),
    pct_zeros=("casos", lambda s: (s == 0).mean() * 100),
)
ax = axes[1]
ax.bar(seas["week"], seas["mean_casos"], color="#4c72b0", alpha=0.85)
ax.set_title("Perfil Estacional por Semana Epidemiológica (1–52)")
ax.set_xlabel("Semana epidemiológica")
ax.set_ylabel("Casos promedio por cantón")
ax.grid(False)

fig.tight_layout()
fig_temp_path = ROOT_DIR / "data/processed/eda/01_serie_nacional_y_estacionalidad.png"
fig.savefig(fig_temp_path, dpi=150)
plt.grid(False)
plt.show()

## 6. Preparación (sin lags ni leakage)

- Codificación cíclica de la semana epidemiológica (`week_sin`, `week_cos`) para capturar la estacionalidad tropical sin discontinuidades entre la semana 52 y la semana 1.
- Sin columnas `*_lag_*` precalculadas.
- Sin columnas derivadas del target en $t$ (`rr`, `clasi_rr`, `clasi_rr_no_0`).

In [ ]:
angle = 2.0 * np.pi * (df["week"] - 1) / 52.0
df["week_sin"] = np.sin(angle)
df["week_cos"] = np.cos(angle)

leakage_cols = [c for c in ["rr", "clasi_rr", "clasi_rr_no_0"] if c in df.columns]
df_clean = df.drop(columns=leakage_cols).copy()

out_clean = ROOT_DIR / "data/processed/dengue_panel_clean.csv"
out_clean.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(out_clean, index=False)

print(f"Panel limpio guardado en: {out_clean}")
print(f"Dimensiones finales: {df_clean.shape} (eliminadas {len(lag_cols)} cols de lag y {len(leakage_cols)} de leakage)")

## MLflow

In [ ]:
import mlflow

mlruns_dir = (ROOT_DIR / "mlruns").resolve()
mlruns_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir = mlruns_dir / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

tracking_uri = f"sqlite:///{(mlruns_dir / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(tracking_uri)

EXPERIMENT_NAME = "dengue_cr_spatiotemporal"
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifacts_dir.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="00_eda_data_audit"):
    mlflow.set_tags({"stage": "eda", "split_policy": "train<=2023|val=2024|test=2025"})
    mlflow.log_params({
        "n_obs": int(len(df_clean)),
        "n_cantons": int(n_cantons),
        "n_weeks": int(n_weeks),
        "dropped_lag_cols": len(lag_cols),
        "dropped_leakage_cols": ",".join(leakage_cols),
    })
    mlflow.log_metrics({
        "global_pct_zeros": round(float(p0_obs_g * 100), 2),
        "global_mean_casos": round(float(mean_g), 4),
        "global_dispersion_idx": round(float(disp_g), 4),
        "cantons_excess_zero_vs_poisson": int((canton_diag["excess_vs_poisson"] > 0.05).sum()),
        "cantons_excess_zero_vs_negbin": int((canton_diag["excess_vs_negbin"] > 0.05).sum()),
    })
    mlflow.log_artifact(str(fig_zi_path), artifact_path="eda_figures")
    mlflow.log_artifact(str(fig_temp_path), artifact_path="eda_figures")

print("Run '00_eda_data_audit' registrado en:", tracking_uri)

In [ ]:
def eval_metrics(y_true, y_pred, prefix):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    return {
        f"{prefix}_mae": round(float(np.mean(np.abs(y_true - y_pred))), 4),
        f"{prefix}_rmse": round(float(np.sqrt(np.mean((y_true - y_pred) ** 2))), 4),
        f"{prefix}_rmsle": round(float(np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))), 4),
    }

df_clean["pred_t1"] = df_clean.groupby("canton")["casos"].shift(1)
df_clean["pred_t52"] = df_clean.groupby("canton")["casos"].shift(52)
val_mask = df_clean["year"] == 2024
test_mask = df_clean["year"] == 2025

for run_name, col, lag_k in [("00_naive_persistence_t1", "pred_t1", 1), ("00_naive_seasonal_t52", "pred_t52", 52)]:
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({"stage": "baseline_naive", "horizon": "1_week"})
        mlflow.log_params({"model_family": "naive", "lag_weeks": lag_k})
        m_val = eval_metrics(df_clean.loc[val_mask, "casos"], df_clean.loc[val_mask, col], "val")
        m_test = eval_metrics(df_clean.loc[test_mask, "casos"], df_clean.loc[test_mask, col], "test")
        mlflow.log_metrics({**m_val, **m_test})
        print(run_name, "->", m_val, m_test)